# Contrôle des anomalies dans les fichiers Excel historiques

Ce notebook sert uniquement à **voir et comprendre les anomalies** présentes dans les fichiers Excel `Suivi_2020.xlsx` à `Suivi_2026.xlsx`.

Il ne crée pas de CSV finaux et il n’exporte pas de données. Il affiche seulement les anomalies directement dans les cellules du notebook.

Objectif : pour chaque anomalie, afficher :

1. le titre de l’anomalie ;
2. une explication simple ;
3. pourquoi c’est important ;
4. le nombre de cas détectés ;
5. un tableau avec la localisation : fichier, feuille, ligne Excel, colonne Excel, boîte, espèce, semaine, valeur.

Les anomalies détectées ici ne sont pas toutes des erreurs à supprimer. Certaines sont des **cas métier à valider**.

## Cellule 1 — Importer les bibliothèques et trouver les fichiers Excel

Cette cellule prépare le notebook.

Elle cherche les fichiers Excel dans le dossier du projet. Si le notebook est lancé depuis un autre endroit, elle essaie aussi le chemin habituel du projet sur le Mac.

In [64]:
from pathlib import Path
import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from IPython.display import display, Markdown

# Nombre de lignes affichées par anomalie.
# Tu peux augmenter si tu veux voir plus de cas.
NB_LIGNES_AFFICHER = 100

# Recherche automatique du dossier contenant les fichiers Excel.
candidats = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/Users/akkouh/Desktop/polybase'),
    Path('/mnt/data'),
]

PROJECT_DIR = None
for dossier in candidats:
    if (dossier / 'Suivi_2020.xlsx').exists():
        PROJECT_DIR = dossier
        break

if PROJECT_DIR is None:
    raise FileNotFoundError("Impossible de trouver les fichiers Suivi_2020.xlsx à Suivi_2026.xlsx. Vérifie que le notebook est dans le bon dossier.")

excel_files = [
    PROJECT_DIR / 'Suivi_2020.xlsx',
    PROJECT_DIR / 'Suivi_2021.xlsx',
    PROJECT_DIR / 'Suivi_2022.xlsx',
    PROJECT_DIR / 'Suivi_2023.xlsx',
    PROJECT_DIR / 'Suivi_2024.xlsx',
    PROJECT_DIR / 'Suivi_2025.xlsx',
    PROJECT_DIR / 'Suivi_2026.xlsx',
]

display(Markdown(f"**Dossier utilisé :** `{PROJECT_DIR}`"))

verification = pd.DataFrame({
    'fichier': [f.name for f in excel_files],
    'trouve': [f.exists() for f in excel_files]
})
display(verification)

**Dossier utilisé :** `/Users/akkouh/Desktop/polybase`

,fichier,trouve
0,Suivi_2020.xlsx,True
1,Suivi_2021.xlsx,True
2,Suivi_2022.xlsx,True
3,Suivi_2023.xlsx,True
4,Suivi_2024.xlsx,True
5,Suivi_2025.xlsx,True
6,Suivi_2026.xlsx,True


## Cellule 2 — Scanner toutes les cellules utiles

Cette cellule lit les fichiers Excel ligne par ligne.

Elle récupère :

- le fichier ;
- la feuille ;
- la ligne Excel ;
- la colonne Excel ;
- l’espèce ;
- le code boîte ;
- la température ;
- le type de mesure ;
- la semaine ;
- la valeur brute ;
- la couleur de cellule.

Ensuite, les autres cellules du notebook utilisent ces données pour afficher les anomalies.

In [65]:
REGEX_ADDITION = re.compile(r"^\s*\d+(?:\s*\+\s*\d+)+\s*$")
REGEX_NOMBRE = re.compile(r"^\s*-?\d+(?:[,.]\d+)?\s*$")
REGEX_CODE_BOITE = re.compile(r"^[A-Z]{3}-[A-Z0-9?]{3}-[0-9]+(\.[0-9]+)?$")


def valeur_est_vide(valeur):
    return valeur is None or (isinstance(valeur, str) and valeur.strip() == "")


def nettoyer_texte(valeur):
    if valeur_est_vide(valeur):
        return None
    return str(valeur).strip()


def calculer_addition(valeur):
    return sum(int(x.strip()) for x in str(valeur).replace(" ", "").split("+"))


def extraire_code_espece(code_boite):
    if valeur_est_vide(code_boite):
        return None
    return str(code_boite).strip().split("-")[0]


def extraire_code_souche(code_boite):
    if valeur_est_vide(code_boite):
        return None
    code_boite = str(code_boite).strip()
    return code_boite.split(".")[0] if "." in code_boite else code_boite


def couleur_cellule(cell):
    fill = cell.fill
    if fill is None or fill.fill_type is None:
        return None

    fg = fill.fgColor
    if fg is None:
        return None

    if fg.type == "rgb":
        return str(fg.rgb).upper()

    if fg.type == "indexed":
        return f"indexed:{fg.indexed}"

    if fg.type == "theme":
        return f"theme:{fg.theme}:{fg.tint}"

    return str(fg.value).upper() if fg.value is not None else None


def est_cellule_bleue(cell):
    couleur = couleur_cellule(cell)
    if couleur is None:
        return False

    couleur = str(couleur).upper()

    return (
        couleur in [
            "FF00B0F0", "FF00BFFF", "FF99CCFF", "FF9DC3E6",
            "FFBDD7EE", "FFDDEBF7", "FF0070C0", "FF4A86E8"
        ]
        or couleur.endswith("B0F0")
        or couleur.endswith("70C0")
        or couleur.endswith("4A86E8")
        or couleur.endswith("9DC3E6")
    )


def est_cellule_grise(cell):
    couleur = couleur_cellule(cell)
    if couleur is None:
        return False

    couleur = str(couleur).upper()

    return (
        couleur in [
            "FFD9D9D9", "FFBFBFBF", "FFA6A6A6", "FF808080",
            "FFE7E6E6", "FFB7B7B7", "FF999999"
        ]
        or couleur.endswith("D9D9D9")
        or couleur.endswith("BFBFBF")
        or couleur.endswith("A6A6A6")
        or couleur.endswith("999999")
    )


def semaine_invalide(semaine):
    if pd.isna(semaine):
        return True
    try:
        semaine_int = int(float(semaine))
        return semaine_int < 1 or semaine_int > 53
    except Exception:
        return True


lignes_excel = []
cellules_releves = []

for excel_path in excel_files:
    if not excel_path.exists():
        continue

    wb = load_workbook(excel_path, data_only=False)

    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]

        # Structure attendue :
        # Ligne 1 : mois
        # Ligne 2 : semaine
        # Colonne A : espèce
        # Colonne B : code boîte
        # Colonne C : température
        # Colonne D : type de mesure
        # Colonnes E et + : valeurs de relevés
        if ws.max_row < 3 or ws.max_column < 5:
            continue

        for row in range(3, ws.max_row + 1):
            espece = nettoyer_texte(ws.cell(row=row, column=1).value)
            code_boite = nettoyer_texte(ws.cell(row=row, column=2).value)
            temperature = nettoyer_texte(ws.cell(row=row, column=3).value)
            type_mesure = nettoyer_texte(ws.cell(row=row, column=4).value)

            if espece is None and code_boite is None and type_mesure is None:
                continue

            code_espece = extraire_code_espece(code_boite)
            code_souche = extraire_code_souche(code_boite)

            lignes_excel.append({
                'fichier': excel_path.name,
                'feuille': sheet_name,
                'ligne_excel': row,
                'espece': espece,
                'code_boite': code_boite,
                'code_souche': code_souche,
                'code_espece': code_espece,
                'temperature': temperature,
                'type_mesure': type_mesure,
            })

            for col in range(5, ws.max_column + 1):
                cell = ws.cell(row=row, column=col)
                valeur = nettoyer_texte(cell.value)
                mois = ws.cell(row=1, column=col).value
                semaine = ws.cell(row=2, column=col).value
                colonne_excel = get_column_letter(col)

                cellules_releves.append({
                    'fichier': excel_path.name,
                    'feuille': sheet_name,
                    'ligne_excel': row,
                    'colonne_excel': colonne_excel,
                    'mois': mois,
                    'semaine': semaine,
                    'espece': espece,
                    'code_boite': code_boite,
                    'code_souche': code_souche,
                    'code_espece': code_espece,
                    'temperature': temperature,
                    'type_mesure': type_mesure,
                    'valeur_brute': valeur,
                    'couleur': couleur_cellule(cell),
                    'cellule_bleue': est_cellule_bleue(cell),
                    'cellule_grise': est_cellule_grise(cell),
                })

lignes_df = pd.DataFrame(lignes_excel)
releves_df = pd.DataFrame(cellules_releves)

display(Markdown("# Scan terminé"))
display(Markdown(f"Nombre de lignes Excel analysées : **{len(lignes_df)}**"))
display(Markdown(f"Nombre de cellules de relevés analysées : **{len(releves_df)}**"))

# Scan terminé

Nombre de lignes Excel analysées : **4582**

Nombre de cellules de relevés analysées : **242896**

## Cellule 3 — Construire tous les tableaux d’anomalies

Cette cellule prépare les tableaux utilisés pour l’affichage.

Elle détecte toutes les anomalies visibles dans les fichiers Excel.

In [66]:
# 1. Valeurs additionnées
anomalie_additions = releves_df[
    releves_df['valeur_brute'].astype(str).str.match(REGEX_ADDITION, na=False)
].copy()

if not anomalie_additions.empty:
    anomalie_additions['valeur_calculee'] = anomalie_additions['valeur_brute'].apply(calculer_addition)


# 2. Points d’interrogation
anomalie_points_interrogation = releves_df[
    releves_df['valeur_brute'].astype(str).str.contains(r"\?", regex=True, na=False)
].copy()


# 3. Autres valeurs non numériques
anomalie_valeurs_non_numeriques = releves_df[
    releves_df['valeur_brute'].notna()
    & ~releves_df['valeur_brute'].astype(str).str.match(REGEX_NOMBRE, na=False)
    & ~releves_df['valeur_brute'].astype(str).str.match(REGEX_ADDITION, na=False)
    & ~releves_df['valeur_brute'].astype(str).str.contains(r"\?", regex=True, na=False)
].copy()


# 4. Températures manquantes
anomalie_temperatures_manquantes = lignes_df[
    lignes_df['code_boite'].notna()
    & lignes_df['temperature'].isna()
].copy()


# 5. Températures non numériques
anomalie_temperatures_non_numeriques = lignes_df[
    lignes_df['temperature'].notna()
    & ~lignes_df['temperature'].astype(str).str.match(REGEX_NOMBRE, na=False)
].copy()


# 6. Cellules bleues
anomalie_cellules_bleues = releves_df[
    releves_df['cellule_bleue'] == True
].copy()


# 7. Cellules grises
anomalie_cellules_grises = releves_df[
    releves_df['cellule_grise'] == True
].copy()


# 8. Codes boîte non standards
anomalie_codes_non_standards = lignes_df[
    lignes_df['code_boite'].notna()
    & ~lignes_df['code_boite'].astype(str).str.match(REGEX_CODE_BOITE, na=False)
].copy()


# 9. Lignes avec espèce mais sans code boîte
anomalie_espece_sans_code_boite = lignes_df[
    lignes_df['espece'].notna()
    & lignes_df['code_boite'].isna()
].copy()


# 10. Même code espèce avec plusieurs noms
codes_espece_ambigus = (
    lignes_df[['code_espece', 'espece']]
    .dropna()
    .drop_duplicates()
    .groupby('code_espece')['espece']
    .nunique()
    .loc[lambda x: x > 1]
    .index
)

anomalie_code_espece_plusieurs_noms = (
    lignes_df[lignes_df['code_espece'].isin(codes_espece_ambigus)]
    .drop_duplicates(subset=['code_espece', 'espece', 'fichier', 'feuille', 'ligne_excel'])
    .sort_values(['code_espece', 'espece', 'fichier', 'ligne_excel'])
)


# 11. Même nom scientifique avec plusieurs codes espèce
noms_especes_ambigus = (
    lignes_df[['espece', 'code_espece']]
    .dropna()
    .drop_duplicates()
    .groupby('espece')['code_espece']
    .nunique()
    .loc[lambda x: x > 1]
    .index
)

anomalie_nom_plusieurs_codes_espece = (
    lignes_df[lignes_df['espece'].isin(noms_especes_ambigus)]
    .drop_duplicates(subset=['espece', 'code_espece', 'fichier', 'feuille', 'ligne_excel'])
    .sort_values(['espece', 'code_espece', 'fichier', 'ligne_excel'])
)


# 12. Même code souche avec plusieurs espèces
codes_souches_ambigus = (
    lignes_df[['code_souche', 'espece']]
    .dropna()
    .drop_duplicates()
    .groupby('code_souche')['espece']
    .nunique()
    .loc[lambda x: x > 1]
    .index
)

anomalie_souche_plusieurs_especes = (
    lignes_df[lignes_df['code_souche'].isin(codes_souches_ambigus)]
    .drop_duplicates(subset=['code_souche', 'espece', 'fichier', 'feuille', 'ligne_excel'])
    .sort_values(['code_souche', 'espece', 'fichier', 'ligne_excel'])
)


# 13. Même code boîte avec plusieurs espèces
codes_boites_ambigus = (
    lignes_df[['code_boite', 'espece']]
    .dropna()
    .drop_duplicates()
    .groupby('code_boite')['espece']
    .nunique()
    .loc[lambda x: x > 1]
    .index
)

anomalie_boite_plusieurs_especes = (
    lignes_df[lignes_df['code_boite'].isin(codes_boites_ambigus)]
    .drop_duplicates(subset=['code_boite', 'espece', 'fichier', 'feuille', 'ligne_excel'])
    .sort_values(['code_boite', 'espece', 'fichier', 'ligne_excel'])
)


# 14. Semaines invalides ou absentes, uniquement quand il y a une valeur dans la cellule
anomalie_semaines_invalides = releves_df[
    releves_df['valeur_brute'].notna()
    & releves_df['semaine'].apply(semaine_invalide)
].copy()


resume_anomalies = pd.DataFrame({
    'anomalie': [
        'Valeurs additionnées',
        'Points d’interrogation',
        'Autres valeurs non numériques',
        'Températures manquantes',
        'Températures non numériques',
        'Cellules bleues',
        'Cellules grises',
        'Codes boîte non standards',
        'Espèce renseignée mais code boîte absent',
        'Même code espèce avec plusieurs noms',
        'Même nom scientifique avec plusieurs codes espèce',
        'Même code souche avec plusieurs espèces',
        'Même code boîte avec plusieurs espèces',
        'Semaines invalides ou absentes',
    ],
    'nombre_lignes_ou_cellules': [
        len(anomalie_additions),
        len(anomalie_points_interrogation),
        len(anomalie_valeurs_non_numeriques),
        len(anomalie_temperatures_manquantes),
        len(anomalie_temperatures_non_numeriques),
        len(anomalie_cellules_bleues),
        len(anomalie_cellules_grises),
        len(anomalie_codes_non_standards),
        len(anomalie_espece_sans_code_boite),
        len(anomalie_code_espece_plusieurs_noms),
        len(anomalie_nom_plusieurs_codes_espece),
        len(anomalie_souche_plusieurs_especes),
        len(anomalie_boite_plusieurs_especes),
        len(anomalie_semaines_invalides),
    ]
})

display(Markdown('# Résumé global des anomalies détectées'))
display(resume_anomalies)

# Résumé global des anomalies détectées

,anomalie,nombre_lignes_ou_cellules
0,Valeurs additionnées,0
1,Points d’interrogation,2
2,Autres valeurs non numériques,2
3,Températures manquantes,3
4,Températures non numériques,2
5,Cellules bleues,4717
6,Cellules grises,50718
7,Codes boîte non standards,31
8,Espèce renseignée mais code boîte absent,1
9,Même code espèce avec plusieurs noms,235


## Cellule 4 — Fonction d’affichage

Cette cellule crée une fonction pour afficher chaque anomalie avec le même format :

- titre ;
- explication ;
- importance ;
- nombre de cas ;
- tableau de localisation.

In [67]:
def afficher_bloc_anomalie(numero, titre, explication, pourquoi, dataframe, nb_lignes=NB_LIGNES_AFFICHER):
    display(Markdown(f"# {numero}. {titre}"))
    display(Markdown(explication))
    display(Markdown(f"**Pourquoi c’est important :** {pourquoi}"))
    display(Markdown(f"**Nombre de cas détectés : {len(dataframe)}**"))

    if dataframe.empty:
        display(Markdown("Aucun cas détecté."))
    else:
        display(Markdown(f"Affichage des **{min(nb_lignes, len(dataframe))}** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus."))
        display(dataframe.head(nb_lignes))

## Cellule 5 — Valeurs additionnées

Cette cellule affiche l’anomalie : **Valeurs additionnées**.

In [68]:
afficher_bloc_anomalie(
    1,
    'Valeurs additionnées',
    'Certaines cellules de relevés contiennent une addition écrite directement dans Excel, par exemple `5+4`, `6+4` ou `200+200`.\n\nD’après la réunion métier, ces valeurs peuvent correspondre à des comptages répartis sur plusieurs zones ou supports.\nElles doivent donc être additionnées pour obtenir le total.',
    'Si elles ne sont pas traitées, Python ne les reconnaît pas comme des nombres.',
    anomalie_additions
)

# 1. Valeurs additionnées

Certaines cellules de relevés contiennent une addition écrite directement dans Excel, par exemple `5+4`, `6+4` ou `200+200`.

D’après la réunion métier, ces valeurs peuvent correspondre à des comptages répartis sur plusieurs zones ou supports.
Elles doivent donc être additionnées pour obtenir le total.

**Pourquoi c’est important :** Si elles ne sont pas traitées, Python ne les reconnaît pas comme des nombres.

**Nombre de cas détectés : 0**

Aucun cas détecté.

## Cellule 6 — Points d’interrogation dans les relevés

Cette cellule affiche l’anomalie : **Points d’interrogation dans les relevés**.

In [69]:
afficher_bloc_anomalie(
    2,
    'Points d’interrogation dans les relevés',
    'Certaines cellules contiennent un point d’interrogation, par exemple `?` ou `10?`.\n\nCela signifie que la valeur est incertaine.',
    'Un point d’interrogation ne doit pas être interprété comme 0 ni comme une valeur sûre.',
    anomalie_points_interrogation
)

# 2. Points d’interrogation dans les relevés

Certaines cellules contiennent un point d’interrogation, par exemple `?` ou `10?`.

Cela signifie que la valeur est incertaine.

**Pourquoi c’est important :** Un point d’interrogation ne doit pas être interprété comme 0 ni comme une valeur sûre.

**Nombre de cas détectés : 2**

Affichage des **2** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,colonne_excel,mois,semaine,espece,code_boite,code_souche,code_espece,temperature,type_mesure,valeur_brute,couleur,cellule_bleue,cellule_grise
176074,Suivi_2025.xlsx,Hydrozoa,133,AA,None,22.0,Kirchenpaueria pinnata,KPI-NBE-1.01,KPI-NBE-1,KPI,10,Nb polypes,?,None,False,False
180208,Suivi_2025.xlsx,Hydrozoa,211,AA,None,22.0,Thyroscyphus sp.1 (white),THY-FGU-1.01,THY-FGU-1,THY,25,Nb polypes,?,None,False,False


## Cellule 7 — Autres valeurs non numériques

Cette cellule affiche l’anomalie : **Autres valeurs non numériques**.

In [70]:
afficher_bloc_anomalie(
    3,
    'Autres valeurs non numériques',
    'Certaines cellules contiennent des valeurs qui ne sont ni des nombres, ni des additions, ni des points d’interrogation.\n\nExemples possibles : `2°°`, `#`, texte, symbole.',
    'Ces valeurs doivent être vérifiées manuellement avant import.',
    anomalie_valeurs_non_numeriques
)

# 3. Autres valeurs non numériques

Certaines cellules contiennent des valeurs qui ne sont ni des nombres, ni des additions, ni des points d’interrogation.

Exemples possibles : `2°°`, `#`, texte, symbole.

**Pourquoi c’est important :** Ces valeurs doivent être vérifiées manuellement avant import.

**Nombre de cas détectés : 2**

Affichage des **2** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,colonne_excel,mois,semaine,espece,code_boite,code_souche,code_espece,temperature,type_mesure,valeur_brute,couleur,cellule_bleue,cellule_grise
36131,Suivi_2021.xlsx,Rhizostomae,47,AV,NOVEMBRE,44.0,None,CTU-FCF-2.02,CTU-FCF-2,CTU,20,Nb polypes,2°°,FF00FF00,False,False
95457,Suivi_2023.xlsx,Hydrozoa,36,AP,None,38.0,None,None,None,None,None,Nb éphyrules,#,None,False,False


## Cellule 8 — Températures manquantes

Cette cellule affiche l’anomalie : **Températures manquantes**.

In [71]:
afficher_bloc_anomalie(
    4,
    'Températures manquantes',
    'Certaines lignes possèdent un code de boîte, mais aucune température cible n’est renseignée.',
    'Sans température, on ne peut pas associer correctement la boîte à une zone thermique.',
    anomalie_temperatures_manquantes
)

# 4. Températures manquantes

Certaines lignes possèdent un code de boîte, mais aucune température cible n’est renseignée.

**Pourquoi c’est important :** Sans température, on ne peut pas associer correctement la boîte à une zone thermique.

**Nombre de cas détectés : 3**

Affichage des **3** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,espece,code_boite,code_souche,code_espece,temperature,type_mesure
3882,Suivi_2025.xlsx,UMR_MARBEC,3,Rhizostoma pulmo,RPU-FBA-1.05,RPU-FBA-1,RPU,None,Nb polypes
3886,Suivi_2025.xlsx,UMR_MARBEC,7,None,RPU-FBA-1.07,RPU-FBA-1,RPU,None,Nb polypes
3890,Suivi_2025.xlsx,UMR_MARBEC,11,None,RPU-FBA-1.08,RPU-FBA-1,RPU,None,Nb polypes


## Cellule 9 — Températures non numériques

Cette cellule affiche l’anomalie : **Températures non numériques**.

In [72]:
afficher_bloc_anomalie(
    5,
    'Températures non numériques',
    'Certaines températures contiennent du texte au lieu d’une valeur numérique.\n\nExemple possible : `MARBEC`.',
    'Ces lignes ne peuvent pas être reliées automatiquement à une zone thermique.',
    anomalie_temperatures_non_numeriques
)

# 5. Températures non numériques

Certaines températures contiennent du texte au lieu d’une valeur numérique.

Exemple possible : `MARBEC`.

**Pourquoi c’est important :** Ces lignes ne peuvent pas être reliées automatiquement à une zone thermique.

**Nombre de cas détectés : 2**

Affichage des **2** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,espece,code_boite,code_souche,code_espece,temperature,type_mesure
3120,Suivi_2025.xlsx,Aurelia,11,Aurelia coerulea,ACA-FBA-1.01,ACA-FBA-1,ACA,MARBEC,Nb polypes
3122,Suivi_2025.xlsx,Aurelia,13,None,ACA-FBA-1.02,ACA-FBA-1,ACA,MARBEC,Nb polypes


## Cellule 10 — Cellules bleues

Cette cellule affiche l’anomalie : **Cellules bleues**.

In [73]:
afficher_bloc_anomalie(
    6,
    "Cellules bleues — absences de suivi justifiées",
    """
Les cellules bleues ne sont pas considérées comme des anomalies.

Elles correspondent à des périodes où le suivi n’a pas été réalisé, notamment pendant la période Covid.
Ce sont donc des valeurs manquantes justifiées.
""",
    """
Il ne faut pas les remplacer par 0.
Un 0 signifie qu’un comptage a été fait et que le résultat est nul.
Une cellule bleue signifie qu’il n’y a pas eu de suivi fiable pour cette semaine.
""",
    anomalie_cellules_bleues
)

# 6. Cellules bleues — absences de suivi justifiées


Les cellules bleues ne sont pas considérées comme des anomalies.

Elles correspondent à des périodes où le suivi n’a pas été réalisé, notamment pendant la période Covid.
Ce sont donc des valeurs manquantes justifiées.


**Pourquoi c’est important :** 
Il ne faut pas les remplacer par 0.
Un 0 signifie qu’un comptage a été fait et que le résultat est nul.
Une cellule bleue signifie qu’il n’y a pas eu de suivi fiable pour cette semaine.


**Nombre de cas détectés : 4717**

Affichage des **100** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,colonne_excel,mois,semaine,espece,code_boite,code_souche,code_espece,temperature,type_mesure,valeur_brute,couleur,cellule_bleue,cellule_grise
11,Suivi_2020.xlsx,Aurelia,3,P,None,12.0,Aurelia labiata,ALA-JKA-1.02,ALA-JKA-1,ALA,5,Nb polypes,None,FF0070C0,True,False
12,Suivi_2020.xlsx,Aurelia,3,Q,None,13.0,Aurelia labiata,ALA-JKA-1.02,ALA-JKA-1,ALA,5,Nb polypes,None,FF0070C0,True,False
13,Suivi_2020.xlsx,Aurelia,3,R,AVRIL,14.0,Aurelia labiata,ALA-JKA-1.02,ALA-JKA-1,ALA,5,Nb polypes,None,FF0070C0,True,False
15,Suivi_2020.xlsx,Aurelia,3,T,None,16.0,Aurelia labiata,ALA-JKA-1.02,ALA-JKA-1,ALA,5,Nb polypes,None,FF0070C0,True,False
16,Suivi_2020.xlsx,Aurelia,3,U,None,17.0,Aurelia labiata,ALA-JKA-1.02,ALA-JKA-1,ALA,5,Nb polypes,None,FF0070C0,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,Suivi_2020.xlsx,Aurelia,12,V,None,18.0,None,None,None,None,None,Nb éphyrules,None,FF0070C0,True,False
487,Suivi_2020.xlsx,Aurelia,12,X,None,20.0,None,None,None,None,None,Nb éphyrules,None,FF0070C0,True,False
488,Suivi_2020.xlsx,Aurelia,12,Y,None,21.0,None,None,None,None,None,Nb éphyrules,None,FF0070C0,True,False
489,Suivi_2020.xlsx,Aurelia,12,Z,None,22.0,None,None,None,None,None,Nb éphyrules,None,FF0070C0,True,False


## Cellule 11 — Cellules grises

Cette cellule affiche l’anomalie : **Cellules grises**.

In [74]:
afficher_bloc_anomalie(
    7,
    'Cellules grises',
    'Les cellules grises indiquent généralement que la boîte n’existe pas physiquement sur cette période.\n\nElle peut ne pas encore avoir été créée, ou avoir déjà été supprimée.',
    'Ces cellules ne doivent pas être considérées comme des données biologiques manquantes.',
    anomalie_cellules_grises
)

# 7. Cellules grises

Les cellules grises indiquent généralement que la boîte n’existe pas physiquement sur cette période.

Elle peut ne pas encore avoir été créée, ou avoir déjà été supprimée.

**Pourquoi c’est important :** Ces cellules ne doivent pas être considérées comme des données biologiques manquantes.

**Nombre de cas détectés : 50718**

Affichage des **100** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,colonne_excel,mois,semaine,espece,code_boite,code_souche,code_espece,temperature,type_mesure,valeur_brute,couleur,cellule_bleue,cellule_grise
832,Suivi_2020.xlsx,Aurelia,19,E,JANVIER,1.0,None,ALI-JKA-1.04,ALI-JKA-1,ALI,5,Nb polypes,None,FFBFBFBF,False,True
833,Suivi_2020.xlsx,Aurelia,19,F,None,2.0,None,ALI-JKA-1.04,ALI-JKA-1,ALI,5,Nb polypes,None,FFBFBFBF,False,True
834,Suivi_2020.xlsx,Aurelia,19,G,None,3.0,None,ALI-JKA-1.04,ALI-JKA-1,ALI,5,Nb polypes,None,FFBFBFBF,False,True
835,Suivi_2020.xlsx,Aurelia,19,H,None,4.0,None,ALI-JKA-1.04,ALI-JKA-1,ALI,5,Nb polypes,None,FFBFBFBF,False,True
836,Suivi_2020.xlsx,Aurelia,19,I,None,5.0,None,ALI-JKA-1.04,ALI-JKA-1,ALI,5,Nb polypes,None,FFBFBFBF,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
967,Suivi_2020.xlsx,Aurelia,21,AJ,AOUT,32.0,None,None,None,None,10,Nb polypes,None,FFBFBFBF,False,True
968,Suivi_2020.xlsx,Aurelia,21,AK,None,33.0,None,None,None,None,10,Nb polypes,None,FFBFBFBF,False,True
969,Suivi_2020.xlsx,Aurelia,21,AL,None,34.0,None,None,None,None,10,Nb polypes,None,FFBFBFBF,False,True
970,Suivi_2020.xlsx,Aurelia,21,AM,None,35.0,None,None,None,None,10,Nb polypes,None,FFBFBFBF,False,True


## Cellule 12 — Codes boîte non standards

Cette cellule affiche l’anomalie : **Codes boîte non standards**.

In [75]:
afficher_bloc_anomalie(
    8,
    'Codes boîte non standards',
    'Certains codes de boîte ne respectent pas le format attendu.\n\nFormat attendu : `AAA-BBB-1.01`, par exemple `ACA-FBA-1.01`.',
    'Le script ne peut pas extraire correctement le code espèce, la provenance, la souche et le numéro de boîte.',
    anomalie_codes_non_standards
)

# 8. Codes boîte non standards

Certains codes de boîte ne respectent pas le format attendu.

Format attendu : `AAA-BBB-1.01`, par exemple `ACA-FBA-1.01`.

**Pourquoi c’est important :** Le script ne peut pas extraire correctement le code espèce, la provenance, la souche et le numéro de boîte.

**Nombre de cas détectés : 31**

Affichage des **31** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,espece,code_boite,code_souche,code_espece,temperature,type_mesure
742,Suivi_2021.xlsx,Rhizostomae,95,Rhizostoma octopus,ROC-1.01,ROC-1,ROC,10,Nb polypes
1360,Suivi_2022.xlsx,Rhizostomae,143,Rhizostoma octopus,ROC-1.01,ROC-1,ROC,10,Nb polypes
1794,Suivi_2023.xlsx,Aurelia,47,Aurelia sp.1,ASP-EVA1.01,ASP-EVA1,ASP,10,Nb polypes
2365,Suivi_2023.xlsx,UMR_MARBEC,3,Rhizostoma pulmo,RPU-FBA-1.01 (2B),RPU-FBA-1,RPU,20,Nb polypes
2369,Suivi_2023.xlsx,UMR_MARBEC,7,None,RPU-FBA-1.02 (7),RPU-FBA-1,RPU,20,Nb polypes
2373,Suivi_2023.xlsx,UMR_MARBEC,11,None,RPU-FBA-1.03 (2),RPU-FBA-1,RPU,20,Nb polypes
2377,Suivi_2023.xlsx,UMR_MARBEC,15,None,RPU-FBA-1.04 (8),RPU-FBA-1,RPU,20,Nb polypes
2453,Suivi_2024.xlsx,Aurelia,43,Aurelia sp.1,ASP-EVA1.01,ASP-EVA1,ASP,10,Nb polypes
3166,Suivi_2025.xlsx,Aurelia,57,Aurelia sp.1,ASP-EVA1.01,ASP-EVA1,ASP,10,Nb polypes
3312,Suivi_2025.xlsx,Hydrozoa,121,Hydrozoa sp.,Station n°1,Station n°1,Station n°1,15,Nb polypes


## Cellule 13 — Espèce renseignée mais code boîte absent

Cette cellule affiche l’anomalie : **Espèce renseignée mais code boîte absent**.

In [76]:
afficher_bloc_anomalie(
    9,
    'Espèce renseignée mais code boîte absent',
    'Certaines lignes contiennent une espèce, mais aucun code boîte.\n\nCela peut arriver dans des lignes de commentaire, des titres, ou des zones mal structurées dans le fichier Excel.',
    'Sans code boîte, la ligne ne peut pas être reliée à une souche, une boîte ou un relevé.',
    anomalie_espece_sans_code_boite
)

# 9. Espèce renseignée mais code boîte absent

Certaines lignes contiennent une espèce, mais aucun code boîte.

Cela peut arriver dans des lignes de commentaire, des titres, ou des zones mal structurées dans le fichier Excel.

**Pourquoi c’est important :** Sans code boîte, la ligne ne peut pas être reliée à une souche, une boîte ou un relevé.

**Nombre de cas détectés : 1**

Affichage des **1** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,espece,code_boite,code_souche,code_espece,temperature,type_mesure
4124,Suivi_2026.xlsx,Hydrozoa,161,Pennaria disticha,None,None,None,25,Nb polypes


## Cellule 14 — Même code espèce avec plusieurs noms scientifiques

Cette cellule affiche l’anomalie : **Même code espèce avec plusieurs noms scientifiques**.

In [77]:
afficher_bloc_anomalie(
    10,
    'Même code espèce avec plusieurs noms scientifiques',
    'Un même code espèce est parfois associé à plusieurs noms scientifiques.\n\nCe n’est pas toujours une erreur : le nom scientifique peut évoluer avec le temps.',
    'Il faut une validation métier pour savoir s’il s’agit d’un changement d’identification ou d’une erreur de saisie.',
    anomalie_code_espece_plusieurs_noms
)

# 10. Même code espèce avec plusieurs noms scientifiques

Un même code espèce est parfois associé à plusieurs noms scientifiques.

Ce n’est pas toujours une erreur : le nom scientifique peut évoluer avec le temps.

**Pourquoi c’est important :** Il faut une validation métier pour savoir s’il s’agit d’un changement d’identification ou d’une erreur de saisie.

**Nombre de cas détectés : 235**

Affichage des **100** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,espece,code_boite,code_souche,code_espece,temperature,type_mesure
2648,Suivi_2024.xlsx,Cubozoa,3,Alatina morandini,AMO-JKA-1.06,AMO-JKA-1,AMO,25,Nb polypes
40,Suivi_2020.xlsx,Cubozoa,3,Alatina morandinii,AMO-JKA-1.01,AMO-JKA-1,AMO,25,Nb polypes
510,Suivi_2021.xlsx,Cubozoa,3,Alatina morandinii,AMO-JKA-1.01,AMO-JKA-1,AMO,25,Nb polypes
1060,Suivi_2022.xlsx,Cubozoa,3,Alatina morandinii,AMO-JKA-1.01,AMO-JKA-1,AMO,25,Nb polypes
1956,Suivi_2023.xlsx,Cubozoa,3,Alatina morandinii,AMO-JKA-1.05,AMO-JKA-1,AMO,25,Nb polypes
...,...,...,...,...,...,...,...,...,...
360,Suivi_2020.xlsx,Semaestomeae,59,None,CLA-JKA-1.02,CLA-JKA-1,CLA,20,Nb polypes
412,Suivi_2020.xlsx,Semaestomeae,111,None,CLA-AHE-2.02,CLA-AHE-2,CLA,15,Nb polypes
416,Suivi_2020.xlsx,Semaestomeae,115,None,CLA-EVA-3.01,CLA-EVA-3,CLA,15,Nb polypes
870,Suivi_2021.xlsx,Semaestomeae,67,None,CLA-JKA-1.02,CLA-JKA-1,CLA,20,Nb polypes


## Cellule 15 — Même nom scientifique avec plusieurs codes espèce

Cette cellule affiche l’anomalie : **Même nom scientifique avec plusieurs codes espèce**.

In [78]:
afficher_bloc_anomalie(
    11,
    'Même nom scientifique avec plusieurs codes espèce',
    'Un même nom scientifique peut apparaître avec plusieurs codes espèce différents.\n\nExemple possible : `Obelia sp.` lié à plusieurs codes.',
    'Il faut décider si les codes doivent être fusionnés ou conservés séparément.',
    anomalie_nom_plusieurs_codes_espece
)

# 11. Même nom scientifique avec plusieurs codes espèce

Un même nom scientifique peut apparaître avec plusieurs codes espèce différents.

Exemple possible : `Obelia sp.` lié à plusieurs codes.

**Pourquoi c’est important :** Il faut décider si les codes doivent être fusionnés ou conservés séparément.

**Nombre de cas détectés : 22**

Affichage des **22** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,espece,code_boite,code_souche,code_espece,temperature,type_mesure
3190,Suivi_2025.xlsx,Aurelia,81,Aurelia sp.4,APH-PHI-1.01,APH-PHI-1,APH,30,Nb polypes
3964,Suivi_2026.xlsx,Aurelia,73,Aurelia sp.4,ATH-THA-1.01,ATH-THA-1,ATH,30,Nb polypes
4420,Suivi_2026.xlsx,Semaestomeae,13,"Chrysaora chesapeakei ""white""",CAC-JKA-2.04,CAC-JKA-2,CAC,20,Nb polypes
3690,Suivi_2025.xlsx,Semaestomeae,11,"Chrysaora chesapeakei ""white""",CCP-TPR-1.08,CCP-TPR-1,CCP,20,Nb polypes
3312,Suivi_2025.xlsx,Hydrozoa,121,Hydrozoa sp.,Station n°1,Station n°1,Station n°1,15,Nb polypes
4070,Suivi_2026.xlsx,Hydrozoa,107,Hydrozoa sp.,Station n°1,Station n°1,Station n°1,15,Nb polypes
3314,Suivi_2025.xlsx,Hydrozoa,123,Hydrozoa sp.,Station n°2,Station n°2,Station n°2,15,Nb polypes
4072,Suivi_2026.xlsx,Hydrozoa,109,Hydrozoa sp.,Station n°2,Station n°2,Station n°2,15,Nb polypes
3316,Suivi_2025.xlsx,Hydrozoa,125,Hydrozoa sp.,Station n°3,Station n°3,Station n°3,15,Nb polypes
4074,Suivi_2026.xlsx,Hydrozoa,111,Hydrozoa sp.,Station n°3,Station n°3,Station n°3,15,Nb polypes


## Cellule 16 — Même code souche avec plusieurs espèces

Cette cellule affiche l’anomalie : **Même code souche avec plusieurs espèces**.

In [79]:
afficher_bloc_anomalie(
    12,
    'Même code souche avec plusieurs espèces',
    'Un même code souche apparaît parfois avec plusieurs noms d’espèces.',
    'Il faut savoir si la souche a changé d’identification ou s’il y a une erreur dans les fichiers Excel.',
    anomalie_souche_plusieurs_especes
)

# 12. Même code souche avec plusieurs espèces

Un même code souche apparaît parfois avec plusieurs noms d’espèces.

**Pourquoi c’est important :** Il faut savoir si la souche a changé d’identification ou s’il y a une erreur dans les fichiers Excel.

**Nombre de cas détectés : 143**

Affichage des **100** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,espece,code_boite,code_souche,code_espece,temperature,type_mesure
2648,Suivi_2024.xlsx,Cubozoa,3,Alatina morandini,AMO-JKA-1.06,AMO-JKA-1,AMO,25,Nb polypes
40,Suivi_2020.xlsx,Cubozoa,3,Alatina morandinii,AMO-JKA-1.01,AMO-JKA-1,AMO,25,Nb polypes
510,Suivi_2021.xlsx,Cubozoa,3,Alatina morandinii,AMO-JKA-1.01,AMO-JKA-1,AMO,25,Nb polypes
1060,Suivi_2022.xlsx,Cubozoa,3,Alatina morandinii,AMO-JKA-1.01,AMO-JKA-1,AMO,25,Nb polypes
1956,Suivi_2023.xlsx,Cubozoa,3,Alatina morandinii,AMO-JKA-1.05,AMO-JKA-1,AMO,25,Nb polypes
...,...,...,...,...,...,...,...,...,...
424,Suivi_2020.xlsx,Semaestomeae,123,None,CNO-JKA-1.02,CNO-JKA-1,CNO,20,Nb polypes
946,Suivi_2021.xlsx,Semaestomeae,143,None,CNO-JKA-1.02,CNO-JKA-1,CNO,20,Nb polypes
1656,Suivi_2022.xlsx,Semaestomeae,215,None,CNO-JKA-1.02,CNO-JKA-1,CNO,20,Nb polypes
1660,Suivi_2022.xlsx,Semaestomeae,219,None,CNO-JKA-1.03,CNO-JKA-1,CNO,20,Nb polypes


## Cellule 17 — Même code boîte avec plusieurs espèces

Cette cellule affiche l’anomalie : **Même code boîte avec plusieurs espèces**.

In [80]:
afficher_bloc_anomalie(
    13,
    'Même code boîte avec plusieurs espèces',
    'Un même code boîte apparaît parfois avec plusieurs espèces selon les années.',
    'Cela peut créer une incohérence dans la base, car une boîte doit être liée à une souche claire.',
    anomalie_boite_plusieurs_especes
)

# 13. Même code boîte avec plusieurs espèces

Un même code boîte apparaît parfois avec plusieurs espèces selon les années.

**Pourquoi c’est important :** Cela peut créer une incohérence dans la base, car une boîte doit être liée à une souche claire.

**Nombre de cas détectés : 25**

Affichage des **25** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,espece,code_boite,code_souche,code_espece,temperature,type_mesure
3192,Suivi_2025.xlsx,Aurelia,83,Aurelia sp. THAI,ATH-THA-1.01,ATH-THA-1,ATH,30,Nb polypes
3964,Suivi_2026.xlsx,Aurelia,73,Aurelia sp.4,ATH-THA-1.01,ATH-THA-1,ATH,30,Nb polypes
3944,Suivi_2026.xlsx,Aurelia,53,Aurelia malayensis,AVA-TAI-1.01,AVA-TAI-1,AVA,25,Nb polypes
2465,Suivi_2024.xlsx,Aurelia,55,"Aurelia sp.2 ""Valentine""",AVA-TAI-1.01,AVA-TAI-1,AVA,25,Nb polypes
3174,Suivi_2025.xlsx,Aurelia,65,"Aurelia sp.2 ""Valentine""",AVA-TAI-1.01,AVA-TAI-1,AVA,20,Nb polypes
70,Suivi_2020.xlsx,Hydrozoa,19,Clytia hemisphaerica,CLH-FVI-1.01,CLH-FVI-1,CLH,15,Nb polypes
546,Suivi_2021.xlsx,Hydrozoa,23,Clytia hemispherica,CLH-FVI-1.01,CLH-FVI-1,CLH,15,Nb polypes
1118,Suivi_2022.xlsx,Hydrozoa,31,Clytia hemispherica,CLH-FVI-1.01,CLH-FVI-1,CLH,15,Nb polypes
3808,Suivi_2025.xlsx,Semaestomeae,129,Cyanea nozakii,CNO-JKA-1.02,CNO-JKA-1,CNO,20,Nb polypes
2280,Suivi_2023.xlsx,Semaestomeae,145,Cyanea nozakii Kishinouye,CNO-JKA-1.02,CNO-JKA-1,CNO,20,Nb polypes


## Cellule 18 — Semaines invalides ou absentes

Cette cellule affiche l’anomalie : **Semaines invalides ou absentes**.

In [82]:
# Résumé des fichiers et feuilles concernés
resume_semaines_invalides = (
    anomalie_semaines_invalides[["fichier", "feuille"]]
    .drop_duplicates()
    .sort_values(["fichier", "feuille"])
)

display(Markdown("## Fichiers et feuilles concernés"))
display(resume_semaines_invalides)

afficher_bloc_anomalie(
    14,
    "Erreur d’en-tête Excel : semaine 0 au lieu de semaine 7",
    """
Certaines colonnes ont un numéro de semaine incorrect dans l’en-tête Excel.

Dans les fichiers concernés, la colonne est placée entre la semaine 6 et la semaine 8, mais elle est indiquée comme semaine `0`.
Il s’agit donc probablement d’une erreur de saisie dans l’en-tête : la semaine doit être corrigée en `7`.

Les valeurs de relevés présentes dans cette colonne ne sont pas forcément fausses.
C’est uniquement le numéro de semaine qui empêche de dater correctement les données.
""",
    """
Polypbase a besoin d’un numéro de semaine correct pour dater chaque relevé.
Après correction manuelle de `0` en `7` dans l’Excel, ces lignes ne devraient plus apparaître dans cette catégorie.
""",
    anomalie_semaines_invalides
)

## Fichiers et feuilles concernés

,fichier,feuille
165240,Suivi_2025.xlsx,Aurelia
204312,Suivi_2025.xlsx,Coronatae
182412,Suivi_2025.xlsx,Cubozoa
169162,Suivi_2025.xlsx,Hydrozoa
184850,Suivi_2025.xlsx,Rhizostomae
195134,Suivi_2025.xlsx,Semaestomeae
205796,Suivi_2025.xlsx,UMR_MARBEC
206432,Suivi_2026.xlsx,Aurelia
241624,Suivi_2026.xlsx,Coronatae
220318,Suivi_2026.xlsx,Cubozoa


# 14. Erreur d’en-tête Excel : semaine 0 au lieu de semaine 7


Certaines colonnes ont un numéro de semaine incorrect dans l’en-tête Excel.

Dans les fichiers concernés, la colonne est placée entre la semaine 6 et la semaine 8, mais elle est indiquée comme semaine `0`.
Il s’agit donc probablement d’une erreur de saisie dans l’en-tête : la semaine doit être corrigée en `7`.

Les valeurs de relevés présentes dans cette colonne ne sont pas forcément fausses.
C’est uniquement le numéro de semaine qui empêche de dater correctement les données.


**Pourquoi c’est important :** 
Polypbase a besoin d’un numéro de semaine correct pour dater chaque relevé.
Après correction manuelle de `0` en `7` dans l’Excel, ces lignes ne devraient plus apparaître dans cette catégorie.


**Nombre de cas détectés : 618**

Affichage des **100** premiers cas. Change `NB_LIGNES_AFFICHER` si tu veux en voir plus.

,fichier,feuille,ligne_excel,colonne_excel,mois,semaine,espece,code_boite,code_souche,code_espece,temperature,type_mesure,valeur_brute,couleur,cellule_bleue,cellule_grise
165240,Suivi_2025.xlsx,Aurelia,11,E,2024,NaN,Aurelia coerulea,ACA-FBA-1.01,ACA-FBA-1,ACA,MARBEC,Nb polypes,500,None,False,False
165293,Suivi_2025.xlsx,Aurelia,12,E,2024,NaN,None,None,None,None,None,Nb éphyrules,50,None,False,False
165346,Suivi_2025.xlsx,Aurelia,13,E,2024,NaN,None,ACA-FBA-1.02,ACA-FBA-1,ACA,MARBEC,Nb polypes,500,None,False,False
165399,Suivi_2025.xlsx,Aurelia,14,E,2024,NaN,None,None,None,None,None,Nb éphyrules,150,None,False,False
165982,Suivi_2025.xlsx,Aurelia,25,E,2024,NaN,Aurelia labiata,ALA-JKA-1.08,ALA-JKA-1,ALA,5,Nb polypes,300,None,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180663,Suivi_2025.xlsx,Hydrozoa,220,E,2024,NaN,None,None,None,None,None,Nb éphyrules,2,None,False,False
180716,Suivi_2025.xlsx,Hydrozoa,221,E,2024,NaN,None,TNI-JKA-1.03,TNI-JKA-1,TNI,15,Nb polypes,50,None,False,False
180769,Suivi_2025.xlsx,Hydrozoa,222,E,2024,NaN,None,None,None,None,None,Nb éphyrules,0,None,False,False
181352,Suivi_2025.xlsx,Hydrozoa,233,E,2024,NaN,Turritopsis pacifica,TPA-JHO-1.01,TPA-JHO-1,TPA,10,Nb polypes,2,None,False,False


# Fin du notebook

Ce notebook sert uniquement à afficher les anomalies visibles dans les fichiers Excel.

À ce stade, il n’exporte rien. Il sert à comprendre les données avant de décider quoi corriger, quoi garder et quoi exclure.

In [90]:
# ============================================================
# Export CSV séparé par type d'anomalie
# Fichiers destinés aux encadrants
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/Users/akkouh/Desktop/POLYPBASE")
OUTPUT_DIR = PROJECT_DIR / "anomalies_par_type"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Fonctions utilitaires
# ============================================================

def get_df(nom_variable):
    """
    Récupère un DataFrame s'il existe dans le notebook.
    Si la variable n'existe pas, on retourne un DataFrame vide.
    """
    obj = globals().get(nom_variable)

    if obj is None:
        return pd.DataFrame()

    if not isinstance(obj, pd.DataFrame):
        return pd.DataFrame()

    return obj


def completer_espece_si_vide(df):
    """
    Complète automatiquement les espèces vides à partir de la dernière espèce connue
    dans le même fichier et la même feuille.

    Cela sert pour les fichiers Excel où l'espèce est écrite une seule fois,
    puis les lignes suivantes sont laissées vides visuellement.
    """
    if df is None or df.empty:
        return df

    df = df.copy()

    colonnes_necessaires = {"fichier", "feuille", "ligne_excel", "espece"}

    if not colonnes_necessaires.issubset(df.columns):
        return df

    df = df.sort_values(["fichier", "feuille", "ligne_excel"])

    df["espece"] = (
        df
        .groupby(["fichier", "feuille"])["espece"]
        .ffill()
    )

    return df


def ajouter_code_espece_si_absent(df):
    """
    Ajoute code_espece à partir du code_boite si la colonne n'existe pas.
    Exemple : CAC-JKA-1.01 -> CAC
    """
    if df is None or df.empty:
        return df

    df = df.copy()

    if "code_espece" not in df.columns and "code_boite" in df.columns:
        df["code_espece"] = (
            df["code_boite"]
            .astype(str)
            .str.strip()
            .str.split("-")
            .str[0]
        )

    return df


def ajouter_code_souche_si_absent(df):
    """
    Ajoute code_souche à partir du code_boite si la colonne n'existe pas.
    Exemple : CAC-JKA-1.01 -> CAC-JKA-1
    """
    if df is None or df.empty:
        return df

    df = df.copy()

    if "code_souche" not in df.columns and "code_boite" in df.columns:
        df["code_souche"] = (
            df["code_boite"]
            .astype(str)
            .str.strip()
            .str.split(".")
            .str[0]
        )

    return df


def liste_unique(valeurs):
    """
    Transforme une liste de valeurs en texte lisible sans doublons.
    """
    valeurs = [
        str(v).strip()
        for v in valeurs
        if pd.notna(v) and str(v).strip() != ""
    ]

    return " | ".join(sorted(set(valeurs)))


def liste_unique_limitee(valeurs, limite=10):
    """
    Transforme une liste de valeurs en texte lisible sans doublons,
    avec une limite pour éviter les cellules trop longues.
    """
    valeurs = [
        str(v).strip()
        for v in valeurs
        if pd.notna(v) and str(v).strip() != ""
    ]

    return " | ".join(sorted(set(valeurs))[:limite])


# ============================================================
# Export simple pour les anomalies ligne par ligne
# ============================================================

def exporter_anomalie_csv(df, type_anomalie, nom_fichier):
    """
    Exporte un DataFrame d'anomalies dans un fichier CSV simple.
    Les colonnes correction_encadrant et commentaire_encadrant sont laissées vides.
    """
    if df is None or df.empty:
        print(f"Aucun cas pour : {type_anomalie}")
        return pd.DataFrame()

    df_export = df.copy()

    # Compléter les espèces vides quand c'est possible
    df_export = completer_espece_si_vide(df_export)

    # Colonnes techniques inutiles pour les encadrants
    colonnes_a_supprimer = [
        "couleur",
        "cellule_bleue",
        "cellule_grise",
        "code_souche",
        "code_espece",
        "mois",
        "semaine",
        "action_attendue",
        "explication",
        "decision_encadrant",
        "correction_espece",
        "correction_code_boite",
        "correction_temperature",
        "correction_valeur",
        "espece_originale",
        "espece_completee_auto",
    ]

    df_export = df_export.drop(
        columns=[col for col in colonnes_a_supprimer if col in df_export.columns],
        errors="ignore"
    )

    # Évite de dupliquer la colonne si elle existe déjà
    if "type_anomalie" in df_export.columns:
        df_export = df_export.drop(columns=["type_anomalie"])

    # Ajouter le type d'anomalie
    df_export.insert(0, "type_anomalie", type_anomalie)

    # Colonnes à remplir par les encadrants
    df_export["correction_encadrant"] = ""
    df_export["commentaire_encadrant"] = ""

    # Colonnes finales simples
    colonnes_finales = [
        "type_anomalie",
        "fichier",
        "feuille",
        "ligne_excel",
        "colonne_excel",
        "espece",
        "code_boite",
        "temperature",
        "type_mesure",
        "valeur_brute",
        "correction_encadrant",
        "commentaire_encadrant",
    ]

    colonnes_existantes = [
        col for col in colonnes_finales
        if col in df_export.columns
    ]

    autres_colonnes = [
        col for col in df_export.columns
        if col not in colonnes_existantes
    ]

    df_export = df_export[colonnes_existantes + autres_colonnes]

    chemin_csv = OUTPUT_DIR / nom_fichier

    df_export.to_csv(
        chemin_csv,
        index=False,
        encoding="utf-8-sig",
        sep=";"
    )

    print(f"CSV créé : {chemin_csv} | {len(df_export)} cas")

    return df_export


# ============================================================
# Exports détaillés pour les incohérences code / nom / souche / boîte
# ============================================================

def exporter_resume_code_espece_plusieurs_noms(df, nom_fichier):
    """
    Fichier 04.
    Une ligne par couple code_espece + nom_detecte.
    """
    type_anomalie = "Même code espèce avec plusieurs noms"

    if df is None or df.empty:
        print(f"Aucun cas pour : {type_anomalie}")
        return pd.DataFrame()

    df = completer_espece_si_vide(df)
    df = ajouter_code_espece_si_absent(df)

    resume = (
        df
        .dropna(subset=["code_espece", "espece"])
        .groupby(["code_espece", "espece"])
        .agg(
            fichiers_concernes=("fichier", liste_unique),
            feuilles_concernees=("feuille", liste_unique),
            exemples_codes_boite=("code_boite", liste_unique_limitee),
            nb_lignes_excel=("ligne_excel", "count"),
        )
        .reset_index()
        .rename(columns={"espece": "nom_detecte"})
        .sort_values(["code_espece", "nom_detecte"])
    )

    resume.insert(0, "type_anomalie", type_anomalie)
    resume["correction_encadrant"] = ""
    resume["commentaire_encadrant"] = ""

    chemin_csv = OUTPUT_DIR / nom_fichier

    resume.to_csv(
        chemin_csv,
        index=False,
        encoding="utf-8-sig",
        sep=";"
    )

    print(f"CSV créé : {chemin_csv} | {len(resume)} lignes code/nom")

    return resume


def exporter_resume_nom_plusieurs_codes(df, nom_fichier):
    """
    Fichier 05.
    Une ligne par couple nom_scientifique + code_espece.
    """
    type_anomalie = "Même nom scientifique avec plusieurs codes espèce"

    if df is None or df.empty:
        print(f"Aucun cas pour : {type_anomalie}")
        return pd.DataFrame()

    df = completer_espece_si_vide(df)
    df = ajouter_code_espece_si_absent(df)

    resume = (
        df
        .dropna(subset=["espece", "code_espece"])
        .groupby(["espece", "code_espece"])
        .agg(
            fichiers_concernes=("fichier", liste_unique),
            feuilles_concernees=("feuille", liste_unique),
            exemples_codes_boite=("code_boite", liste_unique_limitee),
            nb_lignes_excel=("ligne_excel", "count"),
        )
        .reset_index()
        .rename(columns={"espece": "nom_scientifique"})
        .sort_values(["nom_scientifique", "code_espece"])
    )

    resume.insert(0, "type_anomalie", type_anomalie)
    resume["correction_encadrant"] = ""
    resume["commentaire_encadrant"] = ""

    chemin_csv = OUTPUT_DIR / nom_fichier

    resume.to_csv(
        chemin_csv,
        index=False,
        encoding="utf-8-sig",
        sep=";"
    )

    print(f"CSV créé : {chemin_csv} | {len(resume)} lignes nom/code")

    return resume


def exporter_resume_souche_plusieurs_especes(df, nom_fichier):
    """
    Fichier 06.
    Une ligne par couple code_souche + espece_detectee.
    """
    type_anomalie = "Même code souche avec plusieurs espèces"

    if df is None or df.empty:
        print(f"Aucun cas pour : {type_anomalie}")
        return pd.DataFrame()

    df = completer_espece_si_vide(df)
    df = ajouter_code_souche_si_absent(df)

    resume = (
        df
        .dropna(subset=["code_souche", "espece"])
        .groupby(["code_souche", "espece"])
        .agg(
            fichiers_concernes=("fichier", liste_unique),
            feuilles_concernees=("feuille", liste_unique),
            exemples_codes_boite=("code_boite", liste_unique_limitee),
            nb_lignes_excel=("ligne_excel", "count"),
        )
        .reset_index()
        .rename(columns={"espece": "espece_detectee"})
        .sort_values(["code_souche", "espece_detectee"])
    )

    resume.insert(0, "type_anomalie", type_anomalie)
    resume["correction_encadrant"] = ""
    resume["commentaire_encadrant"] = ""

    chemin_csv = OUTPUT_DIR / nom_fichier

    resume.to_csv(
        chemin_csv,
        index=False,
        encoding="utf-8-sig",
        sep=";"
    )

    print(f"CSV créé : {chemin_csv} | {len(resume)} lignes souche/espèce")

    return resume


def exporter_resume_boite_plusieurs_especes(df, nom_fichier):
    """
    Fichier 07.
    Une ligne par couple code_boite + espece_detectee.
    """
    type_anomalie = "Même code boîte avec plusieurs espèces"

    if df is None or df.empty:
        print(f"Aucun cas pour : {type_anomalie}")
        return pd.DataFrame()

    df = completer_espece_si_vide(df)

    resume = (
        df
        .dropna(subset=["code_boite", "espece"])
        .groupby(["code_boite", "espece"])
        .agg(
            fichiers_concernes=("fichier", liste_unique),
            feuilles_concernees=("feuille", liste_unique),
            lignes_excel=("ligne_excel", liste_unique_limitee),
            nb_lignes_excel=("ligne_excel", "count"),
        )
        .reset_index()
        .rename(columns={"espece": "espece_detectee"})
        .sort_values(["code_boite", "espece_detectee"])
    )

    resume.insert(0, "type_anomalie", type_anomalie)
    resume["correction_encadrant"] = ""
    resume["commentaire_encadrant"] = ""

    chemin_csv = OUTPUT_DIR / nom_fichier

    resume.to_csv(
        chemin_csv,
        index=False,
        encoding="utf-8-sig",
        sep=";"
    )

    print(f"CSV créé : {chemin_csv} | {len(resume)} lignes boîte/espèce")

    return resume


# ============================================================
# Résumé global
# ============================================================

exports_resume = []


def ajouter_resume(type_anomalie, nom_fichier, df_export):
    if not df_export.empty:
        exports_resume.append({
            "type_anomalie": type_anomalie,
            "fichier_csv": nom_fichier,
            "nombre_de_cas": len(df_export),
        })


# ============================================================
# 01 — Valeurs de relevés à vérifier
# Regroupe :
# - les valeurs avec point d'interrogation
# - les valeurs non numériques
# ============================================================

anomalie_valeurs_releves_a_verifier = pd.concat(
    [
        get_df("anomalie_points_interrogation"),
        get_df("anomalie_valeurs_non_numeriques"),
    ],
    ignore_index=True
).drop_duplicates()

type_anomalie = "Valeurs de relevés à vérifier"
nom_fichier = "01_valeurs_releves_a_verifier.csv"

df_export = exporter_anomalie_csv(
    anomalie_valeurs_releves_a_verifier,
    type_anomalie,
    nom_fichier
)

ajouter_resume(type_anomalie, nom_fichier, df_export)


# ============================================================
# 02 — Températures à vérifier
# Regroupe :
# - les températures manquantes
# - les températures non numériques
# ============================================================

anomalie_temperatures_a_verifier = pd.concat(
    [
        get_df("anomalie_temperatures_manquantes"),
        get_df("anomalie_temperatures_non_numeriques"),
    ],
    ignore_index=True
).drop_duplicates()

type_anomalie = "Températures à vérifier"
nom_fichier = "02_temperatures_a_verifier.csv"

df_export = exporter_anomalie_csv(
    anomalie_temperatures_a_verifier,
    type_anomalie,
    nom_fichier
)

ajouter_resume(type_anomalie, nom_fichier, df_export)


# ============================================================
# 03 — Codes boîte et espèces à vérifier
# Regroupe :
# - les codes boîte non standards
# - les espèces sans code boîte
# ============================================================

anomalie_codes_boite_et_especes_a_verifier = pd.concat(
    [
        get_df("anomalie_codes_non_standards"),
        get_df("anomalie_espece_sans_code_boite"),
    ],
    ignore_index=True
).drop_duplicates()

type_anomalie = "Codes boîte et espèces à vérifier"
nom_fichier = "03_codes_boite_et_especes_a_verifier.csv"

df_export = exporter_anomalie_csv(
    anomalie_codes_boite_et_especes_a_verifier,
    type_anomalie,
    nom_fichier
)

ajouter_resume(type_anomalie, nom_fichier, df_export)


# ============================================================
# 04 — Même code espèce avec plusieurs noms
# Version détaillée : 1 ligne par couple code_espece + nom_detecte
# ============================================================

nom_fichier = "04_code_espece_plusieurs_noms.csv"

df_export = exporter_resume_code_espece_plusieurs_noms(
    get_df("anomalie_code_espece_plusieurs_noms"),
    nom_fichier
)

ajouter_resume("Même code espèce avec plusieurs noms", nom_fichier, df_export)


# ============================================================
# 05 — Même nom scientifique avec plusieurs codes espèce
# Version détaillée : 1 ligne par couple nom_scientifique + code_espece
# ============================================================

nom_fichier = "05_nom_scientifique_plusieurs_codes.csv"

df_export = exporter_resume_nom_plusieurs_codes(
    get_df("anomalie_nom_plusieurs_codes_espece"),
    nom_fichier
)

ajouter_resume("Même nom scientifique avec plusieurs codes espèce", nom_fichier, df_export)


# ============================================================
# 06 — Même code souche avec plusieurs espèces
# Version détaillée : 1 ligne par couple code_souche + espece_detectee
# ============================================================

nom_fichier = "06_code_souche_plusieurs_especes.csv"

df_export = exporter_resume_souche_plusieurs_especes(
    get_df("anomalie_souche_plusieurs_especes"),
    nom_fichier
)

ajouter_resume("Même code souche avec plusieurs espèces", nom_fichier, df_export)


# ============================================================
# 07 — Même code boîte avec plusieurs espèces
# Version détaillée : 1 ligne par couple code_boite + espece_detectee
# ============================================================

nom_fichier = "07_code_boite_plusieurs_especes.csv"

df_export = exporter_resume_boite_plusieurs_especes(
    get_df("anomalie_boite_plusieurs_especes"),
    nom_fichier
)

ajouter_resume("Même code boîte avec plusieurs espèces", nom_fichier, df_export)


# ============================================================
# Création du résumé global
# ============================================================

resume_df = pd.DataFrame(exports_resume)

if resume_df.empty:
    print("Aucun fichier d’anomalie créé.")
else:
    resume_path = OUTPUT_DIR / "00_resume_anomalies.csv"

    resume_df.to_csv(
        resume_path,
        index=False,
        encoding="utf-8-sig",
        sep=";"
    )

    print("Résumé créé :", resume_path)
    display(resume_df)

CSV créé : /Users/akkouh/Desktop/POLYPBASE/anomalies_par_type/01_valeurs_releves_a_verifier.csv | 4 cas
CSV créé : /Users/akkouh/Desktop/POLYPBASE/anomalies_par_type/02_temperatures_a_verifier.csv | 5 cas
CSV créé : /Users/akkouh/Desktop/POLYPBASE/anomalies_par_type/03_codes_boite_et_especes_a_verifier.csv | 32 cas
CSV créé : /Users/akkouh/Desktop/POLYPBASE/anomalies_par_type/04_code_espece_plusieurs_noms.csv | 39 lignes code/nom
CSV créé : /Users/akkouh/Desktop/POLYPBASE/anomalies_par_type/05_nom_scientifique_plusieurs_codes.csv | 14 lignes nom/code
CSV créé : /Users/akkouh/Desktop/POLYPBASE/anomalies_par_type/06_code_souche_plusieurs_especes.csv | 28 lignes souche/espèce
CSV créé : /Users/akkouh/Desktop/POLYPBASE/anomalies_par_type/07_code_boite_plusieurs_especes.csv | 16 lignes boîte/espèce
Résumé créé : /Users/akkouh/Desktop/POLYPBASE/anomalies_par_type/00_resume_anomalies.csv


,type_anomalie,fichier_csv,nombre_de_cas
0,Valeurs de relevés à vérifier,01_valeurs_releves_a_verifier.csv,4
1,Températures à vérifier,02_temperatures_a_verifier.csv,5
2,Codes boîte et espèces à vérifier,03_codes_boite_et_especes_a_verifier.csv,32
3,Même code espèce avec plusieurs noms,04_code_espece_plusieurs_noms.csv,39
4,Même nom scientifique avec plusieurs codes espèce,05_nom_scientifique_plusieurs_codes.csv,14
5,Même code souche avec plusieurs espèces,06_code_souche_plusieurs_especes.csv,28
6,Même code boîte avec plusieurs espèces,07_code_boite_plusieurs_especes.csv,16
